# 00. 실습 환경 준비하기

이 노트북은 튜토리얼 전체에서 쓰는 도구를 설치하고, **제대로 설치되었는지 확인**하는 노트북입니다.
실험을 시작하기 전에 장비를 교정(calibration)하는 단계라고 생각하시면 됩니다.
여기서 마지막 확인 셀까지 문제없이 실행되면, 나머지 노트북(01~08)도 그대로 실행할 수 있습니다.

**학습 목표**
- 이 튜토리얼에서 쓰는 도구(파이썬, Jupyter, Ollama, 두 가지 모델)가 각각 무슨 역할인지 이해합니다.
- Google Colab 또는 내 컴퓨터(로컬) 중 하나에 실습 환경을 준비합니다.
- 설치 확인 셀로 패키지·Ollama 서버·언어모델·임베딩 모델이 모두 동작하는지 확인합니다.
- 내 컴퓨터 사양에 맞게 모델을 바꾸는 방법을 익힙니다.

## 1. 이 튜토리얼에서 쓰는 도구

| 도구 | 한 줄 설명 | 실험실 비유 |
|---|---|---|
| **파이썬(Python)** | 코드를 작성하는 프로그래밍 언어 | 실험 절차를 적는 언어 |
| **Jupyter 노트북** | 설명(글)과 코드, 실행 결과를 한 문서에 담는 도구. 지금 보고 있는 이 파일입니다 | 결과까지 붙여 넣은 실험노트 |
| **Ollama** | 언어모델(LLM)을 **내 컴퓨터에서 무료로** 돌려 주는 프로그램. 인터넷 API·결제·API 키가 필요 없습니다 | 연구실 안에 들여놓은 분석 장비 |
| **LangChain / LangGraph** | LLM·검색·DB를 연결하는 파이썬 라이브러리 | 장비들을 연결하는 배관·배선 |
| **Chroma / SQLite / Kùzu** | 벡터 DB / 관계형 DB / 그래프 DB (02~04에서 배웁니다) | 서로 다른 방식의 시료 보관함 |

### 모델 2종의 역할

Ollama로 두 가지 모델을 내려받아 씁니다. 모델 이름은 `skku_rag/config.py` 한 곳에서 관리하므로, 노트북 코드에는 이름을 직접 쓰지 않습니다.

| 역할 | 기본 모델 | 하는 일 | 크기 |
|---|---|---|---|
| **언어모델 (LLM)** | `qwen3.5:4b` | 질문을 읽고 **문장으로 답을 생성**합니다 | 약 3.4~4 GB |
| **임베딩 모델 (Embedding)** | `embeddinggemma` | 문장을 **숫자 목록(벡터)으로 변환**합니다. 의미가 비슷한 문장은 비슷한 숫자가 됩니다 | 약 0.6 GB |

> `4b`는 "40억(4 billion) 개의 파라미터"라는 뜻입니다. 숫자가 클수록 똑똑하지만 더 많은 메모리와 시간이 필요합니다.

## 2. 방법 A: Google Colab (설치 없이, 권장)

Colab은 구글이 제공하는 **브라우저 속 Jupyter**입니다. 내 컴퓨터에 아무것도 설치하지 않아도 됩니다.

**① GPU 런타임으로 바꾸기** (LLM이 훨씬 빨라집니다)
```
메뉴 [런타임] → [런타임 유형 변경] → 하드웨어 가속기: [T4 GPU] 선택 → [저장]
```

**② 아래 '환경설정' 셀 실행** (셀 왼쪽 ▶ 버튼 또는 `Shift + Enter`)

이 셀은 다음 일을 자동으로 합니다.
```
저장소 내려받기(git clone) → 파이썬 패키지 설치 → Ollama 설치 → Ollama 서버 켜기 → 모델 2종 내려받기
```
처음에는 **5~10분** 걸립니다. 진행 중에 나오는 긴 로그는 정상입니다.

> **주의**: Colab은 일정 시간 사용하지 않거나 창을 닫으면 **런타임이 초기화**됩니다.
> 새로 접속할 때마다 각 노트북의 첫 셀(환경설정)을 다시 실행하세요. 이 셀은 여러 번 실행해도 안전합니다.

## 3. 방법 B: 내 컴퓨터 (로컬 Jupyter)

### ① Ollama 설치와 모델 내려받기
1. https://ollama.com/download 에서 운영체제(Windows / macOS / Linux)에 맞는 설치 파일을 받아 설치합니다.
2. 터미널(Windows는 PowerShell)을 열고 아래 두 줄을 실행합니다.
```bash
ollama pull qwen3.5:4b       # 언어모델 (약 3.4~4 GB)
ollama pull embeddinggemma   # 임베딩 모델 (약 0.6 GB)
```
3. `ollama list` 를 실행해 두 모델이 보이면 성공입니다. Ollama 앱이 켜져 있으면 서버(`localhost:11434`)도 자동으로 실행됩니다.

### ② 파이썬 준비 — **3.12 권장 (3.10~3.13 가능)**
> **Windows / Mac 주의**: 최신 **Python 3.14는 쓰지 마세요.** 그래프 DB인 Kùzu 0.11.3이 Windows·macOS용 3.14 설치 파일을 제공하지 않아 `pip install` 단계에서 실패합니다.

### ③-1 pip로 설치하기 (기본)
```bash
git clone https://github.com/JDeun/skku_rag_tutorials.git   # 또는 GitHub에서 ZIP으로 내려받아 압축 해제
cd skku_rag_tutorials
python -m venv .venv                 # 이 폴더 전용 파이썬 환경(가상환경) 만들기
source .venv/bin/activate            # Windows: .venv\Scripts\activate
pip install -r requirements.txt jupyterlab   # 필요한 패키지 + Jupyter 한 번에 설치
jupyter lab                          # 브라우저에서 Jupyter 열기
```

### ③-2 uv로 설치하기 (더 빠른 대안)
[uv](https://docs.astral.sh/uv/)는 파이썬 버전과 패키지를 한꺼번에 관리해 주는 빠른 도구입니다. 컴퓨터에 파이썬이 없어도 원하는 버전을 자동으로 받아 줍니다.
```bash
# uv 설치 (macOS/Linux)
curl -LsSf https://astral.sh/uv/install.sh | sh
# uv 설치 (Windows PowerShell)
powershell -ExecutionPolicy ByPass -c "irm https://astral.sh/uv/install.ps1 | iex"

cd skku_rag_tutorials
uv venv --python 3.12                # Python 3.12 가상환경 생성 (없으면 자동 다운로드)
uv pip install -r requirements.txt jupyterlab   # 패키지 설치
uv run jupyter lab                   # 가상환경 안의 Jupyter 실행
```

> **가상환경(venv)이란?** 프로젝트마다 따로 쓰는 "전용 시약장"입니다. 다른 프로젝트의 패키지 버전과 섞이지 않게 해 줍니다.
> Jupyter에서 노트북을 열 때 커널이 이 가상환경의 파이썬인지 확인하세요.

## 환경설정

아래 셀은 **모든 노트북의 첫 번째 코드 셀**로 똑같이 들어 있습니다.
Colab에서는 설치를 진행하고, 로컬 Jupyter에서는 아무 일도 하지 않고 넘어갑니다.

In [ ]:
# [Colab 전용] 저장소를 내려받고 패키지·Ollama·모델을 설치합니다. (로컬 Jupyter에서는 자동으로 건너뜁니다)
import os, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/skku_rag_tutorials"):
        !git clone -q https://github.com/JDeun/skku_rag_tutorials.git /content/skku_rag_tutorials
    %cd /content/skku_rag_tutorials
    !bash scripts/colab_setup.sh

## 4. 설치 확인

이제 장비 교정 단계입니다. 아래 셀들을 **위에서부터 차례로** 실행하세요. 각 셀이 무엇을 확인하는지 주석으로 적어 두었습니다.

### 4-1. 파이썬 버전과 패키지 버전

In [ ]:
import sys
from importlib.metadata import version, PackageNotFoundError

# 파이썬 버전 확인 (3.10 ~ 3.13 이어야 합니다)
print("Python", sys.version.split()[0])
if sys.version_info >= (3, 14):
    print("⚠ Python 3.14 에서는 Kùzu 설치가 실패할 수 있습니다. 3.12를 권장합니다.")

# 튜토리얼에서 쓰는 주요 패키지들의 설치 버전을 출력합니다
packages = ["langchain", "langchain-core", "langchain-ollama", "langchain-chroma",
            "langgraph", "chromadb", "kuzu", "pymupdf", "pandas", "numpy"]
for name in packages:
    try:
        print(f"  {name:18s} {version(name)}")
    except PackageNotFoundError:
        print(f"  {name:18s} ❌ 설치 안 됨 → pip install -r requirements.txt")

### 4-2. 튜토리얼 도우미 모듈과 논문 데이터
`skku_rag`는 이 저장소에 들어 있는 작은 도우미 코드입니다. 모델 이름, 폴더 경로, 논문 읽기처럼 **매번 반복되는 준비 작업**만 모아 두었습니다.

In [ ]:
import warnings
warnings.filterwarnings("ignore")  # 실습과 무관한 경고 메시지는 숨깁니다

from skku_rag import LLM_MODEL, EMBED_MODEL, OLLAMA_BASE_URL, PAPER_DIR

print("언어모델  :", LLM_MODEL)
print("임베딩모델:", EMBED_MODEL)
print("Ollama 주소:", OLLAMA_BASE_URL)

# paper_data 폴더 안의 논문 PDF 목록
for pdf in sorted(PAPER_DIR.glob("*.pdf")):
    print("  📄", pdf.name)

### 4-3. Ollama 서버 연결 확인
Ollama 서버에 "어떤 모델을 가지고 있나요?"라고 물어봅니다(`/api/tags`). 필요한 두 모델이 목록에 있어야 합니다.

In [ ]:
import json, urllib.request

try:
    # Ollama 서버에 설치된 모델 목록을 요청합니다
    with urllib.request.urlopen(f"{OLLAMA_BASE_URL}/api/tags", timeout=5) as resp:
        installed = [m["name"] for m in json.load(resp)["models"]]
    print("✅ Ollama 서버 연결 성공")
    for need in [LLM_MODEL, EMBED_MODEL]:
        # 'embeddinggemma' 는 'embeddinggemma:latest' 로 저장되므로 앞부분으로 비교합니다
        ok = any(name == need or name.split(":")[0] == need for name in installed)
        print(f"  {'✅' if ok else '❌'} {need}" + ("" if ok else f"  → 터미널에서 ollama pull {need}"))
except Exception as e:
    print("❌ Ollama 서버에 연결할 수 없습니다:", e)
    print("   → Ollama 앱을 실행하거나 터미널에서 'ollama serve' 를 실행하세요.")

### 4-4. 언어모델에게 한국어로 한 문장 물어보기
`get_llm()`은 설정된 언어모델을 LangChain의 `ChatOllama` 객체로 돌려줍니다. `.invoke(질문)`으로 질문하면 답이 돌아옵니다.
처음 호출할 때는 모델을 메모리에 올리느라 **수십 초** 걸릴 수 있습니다.

In [ ]:
import time
from skku_rag import get_llm

llm = get_llm()  # 설정된 LLM (생각 과정 출력은 꺼져 있음)

start = time.time()
answer = llm.invoke("재료공학을 한 문장으로 소개해 주세요.")  # 질문 보내기
print(answer.content)                                        # 답변 텍스트만 출력
print(f"\n⏱ {time.time() - start:.1f}초")

### 4-5. 임베딩 모델로 문장을 벡터로 바꿔 보기
`get_embeddings()`는 임베딩 모델을 돌려줍니다. `.embed_query(문장)`은 문장 하나를 **숫자 목록(벡터)** 으로 바꿉니다.
벡터의 길이(차원 수)는 모델마다 정해져 있습니다. 이 숫자들이 무슨 의미인지는 03 노트북에서 자세히 다룹니다.

In [ ]:
from skku_rag import get_embeddings

emb = get_embeddings()
vector = emb.embed_query("구리 배선의 비저항은 선폭이 줄어들수록 커진다.")

print("벡터 길이(차원 수):", len(vector))
print("앞 5개 숫자:", [round(v, 4) for v in vector[:5]])

위 네 셀이 모두 ✅ 와 함께 실행되었다면 준비 끝입니다. 🎉 바로 **01_What_is_RAG.ipynb** 로 넘어가세요.

## 5. 모델 바꾸기

기본 모델 `qwen3.5:4b`는 **노트북(8GB RAM)에서도 돌아가도록** 고른 작은 모델입니다. 사양이 좋다면 더 큰 모델이 더 정확하게 답합니다.

| 내 컴퓨터 사양 | 추천 언어모델 | 내려받는 크기 | 비고 |
|---|---|---|---|
| RAM 8GB 이하, GPU 없음 | `qwen3.5:2b` | 약 2.7 GB | 4b가 너무 느릴 때. 답의 품질은 떨어집니다 |
| **RAM 8GB** (일반 노트북), Colab T4 | **`qwen3.5:4b`** (기본값) | 약 3.4~4 GB | 튜토리얼 기준 모델 |
| RAM 16GB 이상 / Apple Silicon 16GB+ / GPU VRAM 8GB+ | `qwen3.5:9b` | 약 6.6 GB | 정확도가 눈에 띄게 좋아집니다 |
| RAM 32GB 이상 / GPU VRAM 24GB 급 | `qwen3.5:27b` | 약 17 GB | 연구실 워크스테이션용 |

> 모델 목록과 크기는 https://ollama.com/library/qwen3.5/tags 에서 확인할 수 있습니다(2026년 10월 기준).
> 임베딩 모델(`embeddinggemma`)은 바꾸지 않는 것을 권장합니다. 바꾸면 03·06에서 만든 벡터 DB를 처음부터 다시 만들어야 합니다.

### 바꾸는 방법 (셋 중 하나)
1. **설정 파일 수정**: `skku_rag/config.py` 의 `LLM_MODEL = os.getenv("SKKU_LLM_MODEL", "qwen3.5:4b")` 에서 `"qwen3.5:4b"` 부분을 바꿉니다.
2. **터미널 환경변수** (Jupyter를 켜기 전에): `export SKKU_LLM_MODEL=qwen3.5:9b` (Windows PowerShell: `$env:SKKU_LLM_MODEL="qwen3.5:9b"`)
3. **노트북 안에서**: `from skku_rag import ...` 보다 **먼저** 아래처럼 설정합니다. Colab에서는 이 줄을 환경설정 셀 맨 위에 넣으면 설치 스크립트가 새 모델까지 자동으로 내려받습니다.

어느 방법이든 로컬에서는 먼저 `ollama pull <모델이름>` 으로 모델을 내려받아야 하고, 이미 import 한 뒤라면 **커널을 재시작**해야 적용됩니다.

In [ ]:
# 예시: 9b 모델로 바꾸고 싶을 때 아래 주석(#)을 지우고, 커널 재시작 후 가장 먼저 실행하세요.
# import os
# os.environ["SKKU_LLM_MODEL"] = "qwen3.5:9b"

## 6. 문제 해결 FAQ

| 증상 | 원인과 해결 |
|---|---|
| `ModuleNotFoundError: No module named 'skku_rag'` | 노트북이 저장소 **최상위 폴더**에서 열리지 않았습니다. `skku_rag_tutorials` 폴더 안에서 `jupyter lab` 을 실행하세요. Colab에서는 환경설정 셀을 다시 실행하세요(`%cd skku_rag_tutorials`). |
| `ModuleNotFoundError: No module named 'langchain_ollama'` 등 | 패키지가 설치되지 않았거나, Jupyter 커널이 다른 파이썬을 쓰고 있습니다. 가상환경을 활성화한 터미널에서 `pip install -r requirements.txt` 후 그 터미널에서 `jupyter lab` 을 실행하세요. |
| `pip install` 중 **kuzu** 설치 실패 | Python 3.14를 쓰고 있을 가능성이 큽니다. Python 3.12로 가상환경을 다시 만드세요(`uv venv --python 3.12`). |
| `Connection refused` / Ollama 서버 연결 실패 | Ollama가 꺼져 있습니다. Ollama 앱을 실행하거나 터미널에서 `ollama serve`. Colab에서는 환경설정 셀을 다시 실행하세요. |
| `model "qwen3.5:4b" not found` | 모델을 내려받지 않았습니다. `ollama pull qwen3.5:4b` (이름 오타도 확인) |
| 답변이 너무 느림 (1분 이상) | 메모리가 부족해 디스크를 쓰고 있을 수 있습니다. 다른 프로그램을 끄거나 `qwen3.5:2b` 로 바꾸세요. Colab은 T4 GPU 런타임인지 확인하세요. 첫 호출은 모델을 불러오느라 원래 느립니다. |
| Colab에서 갑자기 모든 게 사라짐 | 런타임이 초기화된 것입니다. 첫 셀(환경설정)부터 다시 실행하세요. |
| 답이 틀리거나 엉뚱함 | 4B 소형 모델의 한계입니다. 튜토리얼에서는 이것 자체를 교육 포인트로 다룹니다. 더 큰 모델로 바꾸면 좋아집니다. |

## 정리

- **Ollama**는 내 컴퓨터(또는 Colab)에서 무료로 LLM을 돌려 주는 프로그램이고, 우리는 **언어모델**(답 생성)과 **임베딩 모델**(문장 → 벡터) 두 가지를 씁니다.
- 실습 환경은 **Colab(T4 GPU)** 또는 **로컬(Python 3.12 + Ollama)** 중 하나를 고르면 됩니다.
- 모든 노트북의 첫 셀은 Colab에서만 설치를 수행하고, 로컬에서는 건너뜁니다.
- 모델 이름은 `skku_rag/config.py` 또는 환경변수 `SKKU_LLM_MODEL` 로 바꿀 수 있습니다.

## 직접 해보기

1. 4-4 셀의 질문을 `"XRD 분석이 무엇인지 두 문장으로 설명해 주세요."` 로 바꿔 실행해 보세요. 답이 맞는지 여러분의 전공 지식으로 판단해 보세요.
2. 4-5 셀에서 문장을 영어(`"Copper resistivity increases as the line width shrinks."`)로 바꿔 보세요. 벡터 길이는 바뀌나요? (힌트: 길이는 모델이 정합니다.)
3. 터미널에서 `ollama ps` 를 실행해 지금 메모리에 올라와 있는 모델과 그 크기를 확인해 보세요.